# How long do riders usually wait for a pickup?

Say I want to guess the average wait for rides picked up in one zone next Monday at 8 AM. My first guess would be to look at earlier Mondays at 8 AM in that zone. Before trying anything complicated, I want to see how far this simple history gets me.

Wait is the time between request and pickup, in minutes. The [data engineering notebook](data_engineering1_organized_v3.ipynb) explains how I cleaned it. These averages only cover completed trips. Someone who sees a long wait and leaves the app isn't in this table.


## Which months do I use to learn and check the pattern?

I use the prepared hourly table. If it's not downloaded yet, the notebook gets it from the same pinned [Hugging Face snapshot](https://huggingface.co/datasets/glavvrach79/nyc-mobility-digital-twin/tree/5bf35b8304f2c43f34946ca070ac3f642c8bf80d) as the other notebooks.

I learn usual waits from dates before January 2026, and score forecasts on January through June 2026, months the pattern hasn't seen.


In [ ]:
from pathlib import Path

import duckdb

repo_id = "glavvrach79/nyc-mobility-digital-twin"
filename = "data/gold/zone_hour_wait_time.parquet"
data_path = next(
    (
        root / filename
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / filename).is_file()
    ),
    None,
)
if data_path is None:
    from huggingface_hub import hf_hub_download

    data_path = hf_hub_download(
        repo_id=repo_id, filename=filename, repo_type="dataset",
        revision="5bf35b8304f2c43f34946ca070ac3f642c8bf80d", token=False,
    )

con = duckdb.connect()
con.read_parquet(str(data_path)).create_view("wait_time")
split_date = "2026-01-01"

con.execute(
    "SELECT * FROM wait_time ORDER BY hour, location_id LIMIT 5"
).df()

,hour,location_id,avg_wait_seconds,n_trips_with_wait_data
0,2021-01-01,3,405.412500,80
1,2021-01-01,4,241.751825,137
2,2021-01-01,5,589.454545,11
3,2021-01-01,6,444.200000,20
4,2021-01-01,7,235.445783,249


## What if I have no earlier Mondays at 8 AM?

I start with the average for the same zone, weekday and hour. If that combination has no history, I use the zone's average over all its hours. If the zone has no history at all, I use the average across zones.

Every zone-hour gets the same weight, so an hour with 10 trips counts as much as an hour with 100. That's because I'm forecasting the average wait of a zone-hour. If I wanted the average wait per trip, I would weight hours by trips.


In [ ]:
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE train AS
    SELECT
        location_id,
        dayofweek(hour) AS weekday,
        hour(hour) AS hour_of_day,
        avg_wait_seconds / 60.0 AS wait_minutes
    FROM wait_time
    WHERE hour < CAST(? AS TIMESTAMP)
      AND avg_wait_seconds IS NOT NULL
    ''',
    [split_date],
)

con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE predictions AS
    WITH typical_wait AS (
        SELECT location_id, weekday, hour_of_day, avg(wait_minutes) AS prediction
        FROM train
        GROUP BY location_id, weekday, hour_of_day
    ),
    zone_wait AS (
        SELECT location_id, avg(wait_minutes) AS prediction
        FROM train
        GROUP BY location_id
    ),
    city_wait AS (
        SELECT avg(wait_minutes) AS prediction
        FROM train
    )
    SELECT
        w.hour,
        w.location_id,
        w.avg_wait_seconds / 60.0 AS actual_minutes,
        coalesce(t.prediction, z.prediction, city.prediction) AS predicted_minutes,
        city.prediction AS citywide_minutes
    FROM wait_time AS w
    LEFT JOIN typical_wait AS t
        ON w.location_id = t.location_id
        AND dayofweek(w.hour) = t.weekday
        AND hour(w.hour) = t.hour_of_day
    LEFT JOIN zone_wait AS z ON w.location_id = z.location_id
    CROSS JOIN city_wait AS city
    WHERE w.hour >= CAST(? AS TIMESTAMP)
      AND w.avg_wait_seconds IS NOT NULL
    ''',
    [split_date],
)

## Does knowing the zone and time bring me closer?

I compare the zone-and-time forecast with one citywide historical average. MAE tells me how many minutes I missed the hourly average wait by, whether too high or too low.


In [ ]:
con.execute(
    '''
    SELECT
        count(*) AS zone_hours_scored,
        round(avg(abs(actual_minutes - predicted_minutes)), 2) AS zone_time_mae_minutes,
        round(avg(abs(actual_minutes - citywide_minutes)), 2) AS citywide_mae_minutes
    FROM predictions
    '''
).df()

,zone_hours_scored,zone_time_mae_minutes,citywide_mae_minutes
0,1093884,1.03,1.34


### What I learned from the later months

Across **1,093,884 zone-hours**, the zone-and-time baseline missed by **1.03 minutes** on average, while the citywide average missed by **1.34 minutes**. Knowing where and when brings the forecast about **19 seconds** closer. That's my starting point for any wait-time model.

These are errors for the average wait in an hour. One rider's wait can be much shorter or much longer.
